In [10]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

### Load Config

In [12]:
from config import dir_config, main_config

processed_dir = Path(dir_config.data.processed)

mds_updrs_conf = main_config.MDS_UPDRS
metadata = pd.read_csv(Path(processed_dir, "processed_metadata_all_data_accu_60.csv"), encoding="latin1", index_col=None)

In [13]:
def get_valid_vars(item_list, df_columns, label):
    valid_vars = []
    for var in item_list:
        assert var in df_columns, f"Variable {var} not found in DataFrame columns for {label}"
        valid_vars.append(var)
    return valid_vars


# Helper to safely convert to numeric
def to_numeric_df(df):
    return df.apply(pd.to_numeric, errors="coerce")

In [14]:
all_subjects = metadata["subject_id"].unique()
stanford_subjects = metadata[metadata["experiment_site"] == "Stanford"]["subject_id"].unique()
ucla_subjects = metadata[metadata["experiment_site"] == "UCLA"]["subject_id"].unique()
case_western_subjects = metadata[metadata["experiment_site"] == "Case_Western"]["subject_id"].unique()
harvard_subjects = metadata[metadata["experiment_site"] == "Harvard"]["subject_id"].unique()

# Subjects that have at least one treatment session are PD; the rest are HC
pd_subjects = metadata[metadata["treatment"].notna()]["subject_id"].unique()
hc_subjects = [s for s in all_subjects if s not in pd_subjects]

new_pd_subjects = list(set(stanford_subjects) | (set(harvard_subjects) & set(pd_subjects)))
new_pd_subjects_idx = metadata.index[metadata["subject_id"].isin(new_pd_subjects)]

print(f"Total subjects: {len(all_subjects)} (PD: {len(pd_subjects)}, HC: {len(hc_subjects)})")
print(f"Stanford: {len(stanford_subjects)}, UCLA: {len(ucla_subjects)}, Case Western: {len(case_western_subjects)}, Harvard: {len(harvard_subjects)}")
print(f"New-cohort PD subjects: {len(new_pd_subjects)}")
print(f"New-cohort healthy subjects: {len(hc_subjects)}")

Total subjects: 59 (PD: 41, HC: 18)
Stanford: 16, UCLA: 25, Case Western: 4, Harvard: 14
New-cohort PD subjects: 22
New-cohort healthy subjects: 18


In [15]:
relevant_UPDRS_vars = get_valid_vars(mds_updrs_conf.UPDRS_ITEMS, metadata.columns, "UPDRS")
relevant_tremor_vars = get_valid_vars(mds_updrs_conf.TREMOR_ITEMS, metadata.columns, "TREMOR")
relevant_bradykinesia_vars = get_valid_vars(mds_updrs_conf.BRADYKINESIA_ITEMS, metadata.columns, "BRADYKINESIA")
relevant_pigd_vars = get_valid_vars(mds_updrs_conf.PIGD_ITEMS, metadata.columns, "PIGD")

In [16]:
updrs_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_UPDRS_vars])
tremor_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_tremor_vars])
brady_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_bradykinesia_vars])
pigd_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_pigd_vars])

In [17]:
# Compute scores
updrs_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_UPDRS_vars])
tremor_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_tremor_vars])
brady_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_bradykinesia_vars])
pigd_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_pigd_vars])

metadata.loc[new_pd_subjects_idx, "UPDRS"] = np.nansum(updrs_numeric, axis=1)
metadata.loc[new_pd_subjects_idx, "tremor_score"] = np.nanmean(tremor_numeric, axis=1)
metadata.loc[new_pd_subjects_idx, "bradykinesia_score"] = np.nanmean(brady_numeric, axis=1)
metadata.loc[new_pd_subjects_idx, "pigd_score"] = np.nanmean(pigd_numeric, axis=1)

# Initialize ratio columns for all rows before computing
metadata["trem_by_brady"] = np.nan
metadata["trem_by_pigd"] = np.nan

# Compute ratios with division safety
with np.errstate(divide="ignore", invalid="ignore"):
    trem_by_brady = metadata.loc[new_pd_subjects_idx, "tremor_score"] / metadata.loc[new_pd_subjects_idx, "bradykinesia_score"]
    trem_by_pigd = metadata.loc[new_pd_subjects_idx, "tremor_score"] / metadata.loc[new_pd_subjects_idx, "pigd_score"]

# Assign results
metadata.loc[new_pd_subjects_idx, "trem_by_brady"] = trem_by_brady.replace([np.inf, -np.inf], np.nan)
metadata.loc[new_pd_subjects_idx, "trem_by_pigd"] = trem_by_pigd.replace([np.inf, -np.inf], np.nan)

In [ ]:
def get_label(value, thresholds):
    """Map a value onto one of the ordered [low, high] bins defined in the config.

    Bins are tested in config order and the first match wins, so where adjacent bins
    share a boundary the earlier one claims it (with bradykinetic [0, 0.8] and
    intermediate [0.8, 1], a ratio of exactly 0.8 is bradykinetic). Reordering the
    bins in the config therefore changes which side of a shared boundary wins.

    A value outside every bin is assigned to the nearest end bin rather than dropped,
    so an extreme ratio can never become a silent "unknown" category.
    """
    if pd.isna(value):
        return np.nan
    bins = list(thresholds.items())
    for label, (low, high) in bins:
        if low <= value <= high:
            return label
    return bins[0][0] if value < bins[0][1][0] else bins[-1][0]


# Recompute labels from scratch. These columns are read back from the saved CSV at the
# top of the notebook, so without this reset a subject the loop skips would silently
# keep a stale label instead of becoming NA. Object dtype avoids upcasting a float
# column when strings are assigned (FutureWarning on pandas 2.x, an error in 3.x).
metadata["trem_vs_brady_type"] = pd.Series(pd.NA, index=metadata.index, dtype="object")
metadata["pigd_vs_tremor_type"] = pd.Series(pd.NA, index=metadata.index, dtype="object")

for subject in new_pd_subjects:
    subject_indices = metadata.index[metadata["subject_id"] == subject]
    subject_data = metadata.loc[subject_indices]

    off_rows = subject_data[subject_data["treatment"] == "OFF"]

    if off_rows.empty:
        print(f"No OFF-treatment data for subject {subject}. Skipping...")
        continue

    # Subtype from the OFF-medication exam only, so it is independent of medication.
    trem_vs_brady_type = get_label(off_rows["trem_by_brady"].iloc[0], mds_updrs_conf.TREM_VS_BRADY)
    pigd_vs_tremor_type = get_label(off_rows["trem_by_pigd"].iloc[0], mds_updrs_conf.TREM_VS_PIDG)

    metadata.loc[subject_indices, "trem_vs_brady_type"] = trem_vs_brady_type
    metadata.loc[subject_indices, "pigd_vs_tremor_type"] = pigd_vs_tremor_type

# Now create composite columns
metadata["trem_vs_non-trem"] = np.where(
    (metadata["trem_vs_brady_type"] == "tremor") | (metadata["pigd_vs_tremor_type"] == "tremor"),
    "tremor",
    "non-tremor",
)

metadata["pigd_vs_non-pigd"] = np.where(metadata["pigd_vs_tremor_type"] == "pigd", "pigd", "non-pigd")

# np.where has no NA branch, so patients who could not be subtyped (no UPDRS items)
# would otherwise fall through to "non-tremor"/"non-pigd" and be counted as assessed.
typed = metadata["trem_vs_brady_type"].notna() | metadata["pigd_vs_tremor_type"].notna()
metadata.loc[~typed, ["trem_vs_non-trem", "pigd_vs_non-pigd"]] = np.nan

metadata.loc[metadata["is_pd"] == 0, "trem_vs_non-trem"] = np.nan
metadata.loc[metadata["is_pd"] == 0, "pigd_vs_non-pigd"] = np.nan

In [19]:
# calculate total tremor and bradykinesia and intermediate subjects
new_trem_subjects = metadata.loc[metadata["trem_vs_brady_type"] == "tremor"]["subject_id"].unique()
new_brady_subjects = metadata.loc[metadata["trem_vs_brady_type"] == "bradykinetic"]["subject_id"].unique()
new_intermediate_subjects = metadata.loc[metadata["trem_vs_brady_type"] == "intermediate"]["subject_id"].unique()
print(f"  Tremor dominant:         {len(new_trem_subjects)} \t {new_trem_subjects}")
print(f"  Brady dominant:          {len(new_brady_subjects)} \t {new_brady_subjects}")
print(f"  Intermediate:            {len(new_intermediate_subjects)} \t {new_intermediate_subjects}")
print(f"HC subjects:               {len(hc_subjects)} \t {hc_subjects}")


  Tremor dominant:         11 	 ['P3' 'P6' 'P7' 'P11' 'P12' 'P17' 'P18' 'P19' 'P29' 'P31' 'P32']
  Brady dominant:          10 	 ['P1' 'P4' 'P9' 'P13' 'P20' 'P22' 'P23' 'P28' 'P33' 'P34']
  Intermediate:            1 	 ['P24']
HC subjects:               18 	 ['HC1', 'HC3', 'HC6', 'HC7', 'HC8', 'HC9', 'HC12', 'HC13', 'AV', 'BC', 'BF', 'EM', 'ES', 'GF', 'GP', 'JA', 'MRM', 'SY']


In [34]:
# Ensure improvement columns exist
metadata["UPDRS_improvement"] = np.nan
metadata["tremor_improvement"] = np.nan
metadata["bradykinesia_improvement"] = np.nan

for sub in metadata["subject_id"].unique():
    sub_data = metadata[metadata["subject_id"] == sub]
    if sub_data["treatment"].nunique() < 2:
        continue  # skip if both OFF and ON not present

    try:
        off_row = sub_data[sub_data["treatment"] == "OFF"].iloc[0]
        on_row = sub_data[sub_data["treatment"] == "ON"].iloc[0]

        def percent_improvement(off, on):
            return np.nan if off == 0 else (off - on) * 100 / off

        ui = percent_improvement(off_row["UPDRS"], on_row["UPDRS"])
        ti = percent_improvement(off_row["tremor_score"], on_row["tremor_score"])
        bi = percent_improvement(off_row["bradykinesia_score"], on_row["bradykinesia_score"])

        metadata.loc[sub_data.index, "UPDRS_improvement"] = ui
        metadata.loc[sub_data.index, "tremor_improvement"] = ti
        metadata.loc[sub_data.index, "bradykinesia_improvement"] = bi

    except (IndexError, KeyError, TypeError):
        continue  # skip subjects with missing or malformed data

In [35]:
metadata["Were_Diskinesias_Present"] = metadata["Were_Diskinesias_Present"].map({"yes": 1, "no": 0})
metadata["Did_these_Movements_Interfere_with_Ratings"] = metadata["Did_these_Movements_Interfere_with_Ratings"].map({"yes": 1, "no": 0})

In [36]:
metadata.to_csv(Path(processed_dir, "processed_metadata_all_data_accu_60.csv"), index=False)